# Instacart Star Schema with Unity Catalog Constraints

This notebook creates a formal star schema in `workspace.instacart_gold` with:

## Star Schema Structure

* **dim_product** - Product dimension with denormalized hierarchy
  * Primary Key: `product_id`
  * Attributes: product_name, aisle, department, product_hierarchy

* **dim_order** - Order dimension with temporal attributes  
  * Primary Key: `order_id`
  * Attributes: user_id, order_number, day_of_week, hour_of_day, time_bucket

* **fact_order_product** - Fact table (one row per product per order)
  * Composite Primary Key: `(order_id, add_to_cart_order)`
  * Foreign Keys: `product_id` → dim_product, `order_id` → dim_order
  * Measures: reordered, user_id

## Unity Catalog Constraints

Includes primary keys, foreign keys, and NOT NULL constraints for referential integrity validation.

In [0]:

-- Create the instacart_gold schema if it doesn't exist
CREATE SCHEMA IF NOT EXISTS workspace.instacart_gold;

In [0]:

-- DIMENSION: Products with denormalized hierarchy
-- Primary Key: product_id

CREATE OR REPLACE TABLE workspace.instacart_gold.dim_product AS
SELECT 
  p.product_id,
  p.product_name,
  p.aisle_id,
  a.aisle AS aisle_name,
  p.department_id,
  d.department AS department_name,
  CONCAT(d.department, ' / ', a.aisle, ' / ', p.product_name) AS product_hierarchy,
  p.loaded_at
FROM silver_products p
INNER JOIN silver_aisles a ON p.aisle_id = a.aisle_id
INNER JOIN silver_departments d ON p.department_id = d.department_id;

In [0]:

-- DIMENSION: Orders with temporal and behavioral attributes
-- Primary Key: order_id

CREATE OR REPLACE TABLE workspace.instacart_gold.dim_order AS
SELECT 
  order_id,
  user_id,
  eval_set,
  order_number,
  order_dow,
  CASE order_dow
    WHEN 0 THEN 'Sunday'
    WHEN 1 THEN 'Monday'
    WHEN 2 THEN 'Tuesday'
    WHEN 3 THEN 'Wednesday'
    WHEN 4 THEN 'Thursday'
    WHEN 5 THEN 'Friday'
    WHEN 6 THEN 'Saturday'
  END AS day_of_week_name,
  order_hour_of_day,
  CASE 
    WHEN order_hour_of_day BETWEEN 0 AND 5 THEN 'Night (12AM-6AM)'
    WHEN order_hour_of_day BETWEEN 6 AND 11 THEN 'Morning (6AM-12PM)'
    WHEN order_hour_of_day BETWEEN 12 AND 17 THEN 'Afternoon (12PM-6PM)'
    WHEN order_hour_of_day BETWEEN 18 AND 23 THEN 'Evening (6PM-12AM)'
  END AS time_of_day_bucket,
  days_since_prior_order,
  loaded_at
FROM silver_orders;

In [0]:

-- FACT TABLE: Order Products (grain = one row per product per order)
-- Composite Primary Key: (order_id, add_to_cart_order)
-- Foreign Keys: product_id -> dim_product, order_id -> dim_order

CREATE OR REPLACE TABLE workspace.instacart_gold.fact_order_product AS
SELECT 
  op.order_id,
  op.product_id,
  op.add_to_cart_order,
  CAST(op.reordered AS BOOLEAN) AS reordered,
  o.user_id,
  op.source_system,
  op.loaded_at
FROM silver_order_products op
INNER JOIN silver_orders o ON op.order_id = o.order_id
WHERE op.product_id IN (SELECT product_id FROM silver_products);

In [0]:

-- Validates keys and relationships before catalog constraints are registered

WITH product_checks AS (
  SELECT
    COUNT(*) AS row_count,
    COUNT_IF(product_id IS NULL) AS null_keys,
    COUNT(*) - COUNT(DISTINCT product_id) AS duplicate_keys
  FROM workspace.instacart_gold.dim_product
),
order_checks AS (
  SELECT
    COUNT(*) AS row_count,
    COUNT_IF(order_id IS NULL) AS null_keys,
    COUNT(*) - COUNT(DISTINCT order_id) AS duplicate_keys
  FROM workspace.instacart_gold.dim_order
),
fact_checks AS (
  SELECT
    COUNT(*) AS row_count,
    COUNT_IF(
      order_id IS NULL
      OR product_id IS NULL
      OR add_to_cart_order IS NULL
    ) AS null_keys,
    COUNT(*) - COUNT(
      DISTINCT struct(order_id, add_to_cart_order)
    ) AS duplicate_fact_keys
  FROM workspace.instacart_gold.fact_order_product
),
relationship_checks AS (
  SELECT
    COUNT_IF(p.product_id IS NULL) AS unmatched_products,
    COUNT_IF(o.order_id IS NULL) AS unmatched_orders
  FROM workspace.instacart_gold.fact_order_product f
  LEFT JOIN workspace.instacart_gold.dim_product p
    ON f.product_id = p.product_id
  LEFT JOIN workspace.instacart_gold.dim_order o
    ON f.order_id = o.order_id
)
SELECT
  p.row_count AS product_dimension_rows,
  o.row_count AS order_dimension_rows,
  f.row_count AS fact_rows,

  -- Row counts (informational - actual counts from source data)
  CASE 
    WHEN p.row_count > 0 THEN 'PASS'
    ELSE 'FAIL: dim_product is empty'
  END AS product_count_check,

  CASE 
    WHEN o.row_count > 0 THEN 'PASS'
    ELSE 'FAIL: dim_order is empty'
  END AS order_count_check,

  CASE 
    WHEN f.row_count > 0 THEN 'PASS'
    ELSE 'FAIL: fact_order_product is empty'
  END AS fact_count_check,

  assert_true(
    p.null_keys + p.duplicate_keys = 0,
    'dim_product product_id must be nonnull and unique'
  ) AS product_key_check,

  assert_true(
    o.null_keys + o.duplicate_keys = 0,
    'dim_order order_id must be nonnull and unique'
  ) AS order_key_check,

  assert_true(
    f.null_keys + f.duplicate_fact_keys = 0,
    'fact composite key must be nonnull and unique'
  ) AS fact_key_check,

  assert_true(
    r.unmatched_products = 0,
    'all fact product ids must match dim_product'
  ) AS product_relationship_check,

  assert_true(
    r.unmatched_orders = 0,
    'all fact order ids must match dim_order'
  ) AS order_relationship_check

FROM product_checks p
CROSS JOIN order_checks o
CROSS JOIN fact_checks f
CROSS JOIN relationship_checks r;

In [0]:

-- Removes existing named constraints so this query can be safely rerun

ALTER TABLE workspace.instacart_gold.fact_order_product
DROP CONSTRAINT IF EXISTS fact_order_product_product_fk;

ALTER TABLE workspace.instacart_gold.fact_order_product
DROP CONSTRAINT IF EXISTS fact_order_product_order_fk;

ALTER TABLE workspace.instacart_gold.fact_order_product
DROP CONSTRAINT IF EXISTS fact_order_product_pk;

ALTER TABLE workspace.instacart_gold.dim_product
DROP CONSTRAINT IF EXISTS dim_product_pk CASCADE;

ALTER TABLE workspace.instacart_gold.dim_order
DROP CONSTRAINT IF EXISTS dim_order_pk CASCADE;



-- Ensures all primary and foreign key columns are declared not null

ALTER TABLE workspace.instacart_gold.dim_product
ALTER COLUMN product_id SET NOT NULL;

ALTER TABLE workspace.instacart_gold.dim_order
ALTER COLUMN order_id SET NOT NULL;

ALTER TABLE workspace.instacart_gold.fact_order_product
ALTER COLUMN order_id SET NOT NULL;

ALTER TABLE workspace.instacart_gold.fact_order_product
ALTER COLUMN product_id SET NOT NULL;

ALTER TABLE workspace.instacart_gold.fact_order_product
ALTER COLUMN add_to_cart_order SET NOT NULL;



-- Creates the three primary keys

ALTER TABLE workspace.instacart_gold.dim_product
ADD CONSTRAINT dim_product_pk
PRIMARY KEY (product_id);

ALTER TABLE workspace.instacart_gold.dim_order
ADD CONSTRAINT dim_order_pk
PRIMARY KEY (order_id);

ALTER TABLE workspace.instacart_gold.fact_order_product
ADD CONSTRAINT fact_order_product_pk
PRIMARY KEY (order_id, add_to_cart_order);



-- Creates the two fact-to-dimension relationships

ALTER TABLE workspace.instacart_gold.fact_order_product
ADD CONSTRAINT fact_order_product_product_fk
FOREIGN KEY (product_id)
REFERENCES workspace.instacart_gold.dim_product (product_id);

ALTER TABLE workspace.instacart_gold.fact_order_product
ADD CONSTRAINT fact_order_product_order_fk
FOREIGN KEY (order_id)
REFERENCES workspace.instacart_gold.dim_order (order_id);